# HV-011 — Measurement Uncertainty Budget

**Technical gate:** MU-001  
**Dataset release:** VSM-LAB-DATA-v1.1  
**Public specification:** config/public/lab_spec_v1.json

This notebook builds a traceable uncertainty budget for a calibrated 60-second stable RH result at the four declared uncertainty-evaluation points.

## 1. Engineering Question

> What combined and expanded uncertainty is associated with a calibrated 60-second stable RH result, using only analyst-visible evidence and public laboratory specifications?

The notebook separates **residual bias** from **standard uncertainty** and explicitly prevents double-counting.

## 2. Reproducible Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.uncertainty_analysis import (
    build_sensitivity_table,
    build_uncertainty_budget,
    load_frozen_c1_offset,
    load_public_lab_spec,
    load_repeatability_sd,
    load_stability_excursion,
    load_static_and_matrix,
    point_evidence_table,
)

tables_dir = project_root / "reports" / "tables"
figures_dir = project_root / "reports" / "figures"
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")

## 3. MU-01 — Public Specification and Provenance Audit

The downstream repository contains an imported analyst-visible public specification with upstream provenance.

No private simulator truth is used.

In [ ]:
public_spec = load_public_lab_spec(project_root)

ref_spec = public_spec["reference_public_spec"]
dut_spec = public_spec["dut_public_spec"]
chamber_spec = public_spec["chamber_public_spec"]
daq_spec = public_spec["daq_public_spec"]
unc_points = public_spec["experimental_design"]["uncertainty_evaluation_points"]

assert public_spec["lab_identity"]["measurand"] == "relative_humidity"
assert public_spec["lab_identity"]["units"] == "%RH"
assert len(unc_points) == 4
assert np.isclose(
    ref_spec["expanded_calibration_uncertainty_rh_percent"],
    0.40,
)
assert np.isclose(ref_spec["coverage_factor_k"], 2.0)

print("Declared uncertainty-evaluation points")
display(pd.DataFrame(unc_points))

print("Reference expanded calibration uncertainty:",
      ref_spec["expanded_calibration_uncertainty_rh_percent"], "%RH")
print("Reference coverage factor:",
      ref_spec["coverage_factor_k"])

print("MU-01 public specification audit: PASS")

## 4. MU-02 — Frozen Measurement Model

Primary reported result:

```text
Y = mean(RH_DUT over 60 s) + C1
```

Frozen C1:

```text
+1.2266788194 %RH
```

Dynamic measurements are outside this uncertainty model.

In [ ]:
frozen_c1_offset = load_frozen_c1_offset(project_root)
repeatability_sd = load_repeatability_sd(project_root)
stability_excursion = load_stability_excursion(project_root)

u_repeat_mean = repeatability_sd / np.sqrt(60.0)
u_ref_cal = (
    ref_spec["expanded_calibration_uncertainty_rh_percent"]
    / ref_spec["coverage_factor_k"]
)
u_stability = stability_excursion / np.sqrt(3.0)

assert np.isclose(frozen_c1_offset, 1.2266788194444438)

print(f"Frozen C1 offset        : {frozen_c1_offset:.10f} %RH")
print(f"Pooled repeatability SD : {repeatability_sd:.6f} %RH")
print(f"u_repeat_mean           : {u_repeat_mean:.6f} %RH")
print(f"u_ref_cal               : {u_ref_cal:.6f} %RH")
print(f"Stability excursion     : {stability_excursion:.6f} %RH")
print(f"u_stability             : {u_stability:.6f} %RH")

print("MU-02 frozen measurement model: PASS")

## 5. MU-03 — Point-Specific Residual/State Evidence

Evidence role is frozen as:

- 30/25, 50/25, 70/25 → independent **VALIDATION** data;
- 80/35 → independent **CHARACTERIZATION** data.

Each point must contain four Test_IDs corresponding to Run × Direction structure.

Point-level state variability removes the repeatability-of-mean contribution before combination:

```text
u_state = sqrt(max(s_point² - u_repeat_mean², 0))
```

In [ ]:
static_df, test_matrix_df = load_static_and_matrix(project_root)

point_test_evidence_df, point_summary_df = point_evidence_table(
    static_df=static_df,
    test_matrix_df=test_matrix_df,
    public_spec=public_spec,
    c1_offset=frozen_c1_offset,
    repeatability_sd=repeatability_sd,
    samples_per_result=60,
)

assert len(point_summary_df) == 4
assert (point_summary_df["N_Test_IDs"] == 4).all()

print("Test_ID-level evidence")
display(point_test_evidence_df.round(5))

print("Point-level evidence")
display(point_summary_df.round(5))

print("MU-03 point-specific evidence: PASS")

## 6. MU-04 — Primary Uncertainty Budget

Primary standard components:

1. repeatability of the 60-s mean;
2. point-specific residual/state variability;
3. reference calibration uncertainty;
4. bounded observed stability excursion.

Residual bias remains separate.

In [ ]:
budget_df, uncertainty_summary_df = build_uncertainty_budget(
    point_summary_df=point_summary_df,
    public_spec=public_spec,
    stability_excursion=stability_excursion,
    coverage_factor=2.0,
)

display(budget_df.round(6))
print("Point uncertainty summary")
display(
    uncertainty_summary_df[
        [
            "Point_ID",
            "RH_Percent",
            "Temperature_C",
            "Evidence_Role",
            "Point_Residual_Bias",
            "Point_MBE_SD",
            "U_State",
            "U_Reference_Calibration",
            "U_Stability",
            "Combined_Standard_Uncertainty",
            "Expanded_Uncertainty",
            "Abs_Bias_Plus_Expanded_Uncertainty",
        ]
    ].round(5)
)

assert np.isfinite(
    uncertainty_summary_df["Combined_Standard_Uncertainty"]
).all()
assert np.isfinite(
    uncertainty_summary_df["Expanded_Uncertainty"]
).all()

print("MU-04 primary uncertainty budget: PASS")

## 7. MU-05 — Double-Counting Audit

The following are deliberately **not** added as separate primary components:

- DUT resolution / DUT quantization;
- reference resolution / reference quantization;
- reference repeatability;
- HV-005 Run and Day paired shifts;
- HV-010 short-term stability scatter;
- public reference drift limit;
- HV-010 span/shape change.

Reasoning:

- quantization and short-term scatter are already present in empirical repeatability;
- point-specific between-Test_ID variability captures state/run/direction variation;
- observed stability drift already contains analyst-visible DUT-reference temporal behavior;
- span/shape change comes from the same aging campaign and is dependent on that evidence.

In [ ]:
excluded_primary_components = pd.DataFrame(
    [
        ["DUT resolution / quantization", "Embedded in empirical repeatability"],
        ["Reference resolution / quantization", "Embedded in empirical repeatability"],
        ["Reference repeatability", "Embedded in empirical DUT-reference repeatability"],
        ["HV-005 Run/Day shifts", "State/systematic structure; avoid duplication with point variability"],
        ["HV-010 short-term scatter", "Overlaps repeatability"],
        ["Reference drift limit", "Sensitivity only; may overlap observed stability drift"],
        ["HV-010 span/shape change", "Dependent stability evidence; carry as documented condition"],
    ],
    columns=["Candidate_Component", "Primary_Treatment"],
)

display(excluded_primary_components)
print("MU-05 double-counting audit: PASS")

## 8. MU-06 — Sensitivity Cases

Two non-primary sensitivity cases are reported:

### Reference drift

Public annual drift limit:

```text
0.15 %RH/year
```

scaled to 90 days and modeled only for sensitivity.

### Chamber uniformity

Public chamber RH uniformity:

```text
±0.30 %RH
```

The public specification does not define DUT/reference placement or a differential spatial model, so this is not promoted to the primary budget.

In [ ]:
sensitivity_df = build_sensitivity_table(
    uncertainty_summary_df,
    public_spec,
)

display(
    sensitivity_df[
        [
            "Point_ID",
            "Case",
            "Combined_Standard_Uncertainty",
            "Expanded_Uncertainty_k2",
            "Reference_Drift_90d_Bound",
            "U_Reference_Drift_90d",
            "Chamber_Uniformity_Bound",
            "U_Chamber_Uniformity_Sensitivity",
        ]
    ].round(6)
)

print("MU-06 sensitivity analysis: PASS")

## 9. MU-07 — Variance Contribution Diagnostics

The budget is examined in variance space so the dominant uncertainty sources are visible.

In [ ]:
variance_summary_df = (
    budget_df
    .groupby(["Point_ID", "RH_Percent", "Temperature_C"], as_index=False)
    .apply(
        lambda g: g.assign(
            Variance_Fraction=(
                g["Variance_Contribution"]
                / g["Variance_Contribution"].sum()
            )
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)

display(
    variance_summary_df[
        [
            "Point_ID",
            "Component",
            "Standard_Uncertainty",
            "Variance_Fraction",
        ]
    ].round(5)
)

print("MU-07 variance contribution diagnostics: PASS")

## 10. Curated Figures

In [ ]:
# Component standard uncertainty by point
component_pivot = budget_df.pivot(
    index="Point_ID",
    columns="Component",
    values="Standard_Uncertainty",
)

fig, ax = plt.subplots(figsize=(9, 5))
component_pivot.plot(kind="bar", ax=ax)
ax.set_xlabel("Evaluation point")
ax.set_ylabel("Standard uncertainty (%RH)")
ax.set_title("HV-011 Standard Uncertainty Components")
ax.legend(title="Component")
fig.tight_layout()
fig.savefig(
    figures_dir / "hv011_uncertainty_components.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

# Expanded uncertainty by point
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    uncertainty_summary_df["Point_ID"],
    uncertainty_summary_df["Expanded_Uncertainty"],
)
ax.set_xlabel("Evaluation point")
ax.set_ylabel("Expanded uncertainty U, k=2 (%RH)")
ax.set_title("HV-011 Expanded Uncertainty by Evaluation Point")
fig.tight_layout()
fig.savefig(
    figures_dir / "hv011_expanded_uncertainty_by_point.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Curated HV-011 figures exported.")

## 11. Curated Evidence Export

In [ ]:
point_test_evidence_df.to_csv(
    tables_dir / "hv011_uncertainty_point_evidence.csv",
    index=False,
)
budget_df.to_csv(
    tables_dir / "hv011_uncertainty_budget.csv",
    index=False,
)
uncertainty_summary_df.to_csv(
    tables_dir / "hv011_uncertainty_summary.csv",
    index=False,
)
sensitivity_df.to_csv(
    tables_dir / "hv011_uncertainty_sensitivity.csv",
    index=False,
)

print("Curated HV-011 tables exported.")

## 12. Initial MU-001 Checkpoint

A clean-kernel Run All should establish:

- MU-01 public specification audit PASS;
- MU-02 frozen measurement model PASS;
- MU-03 point-specific evidence PASS;
- MU-04 primary uncertainty budget PASS;
- MU-05 double-counting audit PASS;
- MU-06 sensitivity analysis PASS;
- MU-07 variance contribution diagnostics PASS;
- four curated CSV tables;
- two curated figures.

The next step is engineering interpretation and formal MU-001 gate closure.